# Entregável 3 — Análise Estatística Inicial da Base

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.

---

## Objetivo e posição no pipeline

Este notebook cumpre o **Entregável 3** do pipeline da disciplina: **caracterizar estatisticamente a base** *antes* de limpeza profunda (E4), segmentação (E5) e extração de atributos (E6).

| Entregável anterior | O que já foi feito | Ligação a este notebook |
|---------------------|-------------------|-------------------------|
| **E1 — Aquisição** | Caracterização do sinal, Nyquist, metadados, **11 modelos Schiller** no campo `device` | Mesma base `ptbxl_database.csv`; aqui quantificamos **heterogeneidade por dispositivo** (idade, peso, SQI) |
| **E2 — SQI** | Métricas por derivação; ficheiros em `entregavel-2/outputs/` | Ligamos **SQI mediano por `ecg_id`** às variáveis demográficas e ao `device` |

**Perguntas estatísticas centrais:** Distribuições aproximadamente normais? Homogeneidade de variâncias (sexo **e** dispositivo)? Associações entre demografia, **dispositivo** e SQI?

> **Nota:** Idade = 300 anos no PTB-XL codifica idade ≥ 90 (anonimização HIPAA). Tratamos esses casos explicitamente nas análises.


---
## 1. Configuração do ambiente


In [ ]:
# !pip install -r ../../../requirements.txt

import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, kstest, normaltest, levene, bartlett, kruskal

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

CORES = {'primaria': '#1a3a5c', 'secundaria': '#e74c3c', 'ok': '#2ecc71', 'info': '#3498db', 'neutro': '#95a5a6'}

DATA_DIR = Path('../../../data/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3')
FIG_DIR = Path('../figuras')
OUT_DIR = Path('../outputs')
for d in (FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SQI_PATH = Path('../../entregavel-2/outputs/entregavel2_sqi_resultados.csv')
E2_REJEITADOS = Path('../../entregavel-2/outputs/entregavel2_registros_rejeitados.csv')

print('Configuração OK.')
print('  Dados:', DATA_DIR.resolve())
print('  Figuras:', FIG_DIR.resolve())
print('  E2 — SQI completo:', 'OK' if SQI_PATH.exists() else 'ausente')
print('  E2 — rejeitados: ', 'OK' if E2_REJEITADOS.exists() else 'ausente (só informativo)')


---
## 2. Carregamento dos dados e variáveis derivadas

Carregamos `ptbxl_database.csv` e `scp_statements.csv` (mapeamento código SCP → `diagnostic_class` / superclasse), como no **Entregável 1**.


In [ ]:
df = pd.read_csv(DATA_DIR / 'ptbxl_database.csv', index_col='ecg_id')
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

scp_df = pd.read_csv(DATA_DIR / 'scp_statements.csv', index_col=0)
codigo_para_superclasse = scp_df['diagnostic_class'].dropna().to_dict()

def obter_superclasses(scp_dict):
    scs = set()
    for codigo in scp_dict:
        if codigo in codigo_para_superclasse:
            sc = codigo_para_superclasse[codigo]
            if isinstance(sc, str) and sc.strip():
                scs.add(sc)
    return list(scs) if scs else ['ND']

df['superclasses'] = df['scp_codes'].apply(obter_superclasses)

def superclasse_primaria(lista):
    prioridade = ['NORM', 'MI', 'STTC', 'CD', 'HYP', 'ND']
    for p in prioridade:
        if p in lista:
            return p
    return lista[0] if lista else 'ND'

df['sc_primaria'] = df['superclasses'].apply(superclasse_primaria)

# Idade: marcar códigos HIPAA
df['idade_censurada_90plus'] = df['age'] >= 90  # inclui 300
df['age_analise'] = df['age'].replace(300, np.nan)  # excluir da modelagem paramétrica de idade

print(f'Registos: {len(df):,}')
print(df[['age', 'sex', 'height', 'weight']].describe().T.round(2))


---
## 3. Estatística descritiva (metadados)


In [1]:
cols_num = ['age', 'height', 'weight']
desc = df[cols_num].describe(percentiles=[0.25, 0.5, 0.75]).T
desc['IQR'] = desc['75%'] - desc['25%']
desc['missing_n'] = [df[c].isna().sum() for c in cols_num]
desc['missing_%'] = 100 * desc['missing_n'] / len(df)

print('Tabela 1 — Descritiva (valores brutos; idade 300 mantida onde aplicável)')
print(desc.round(3).to_string())

# Sexo: 0 masculino, 1 feminino (convenção PTB-XL)
sex_counts = df['sex'].value_counts().rename(index={0: 'M', 1: 'F'})
print('\nSexo:', sex_counts.to_dict())

prim = df['sc_primaria'].value_counts()
print('\nSuperclasse primária (heurística de prioridade):')
print(prim)


NameError: name 'df' is not defined

---
## 4. Visualizações — histogramas e boxplots

Histogramas com curva KDE para **idade** (excluindo apenas o valor 300 para o eixo mais legível), **peso** e **altura**; boxplot de **idade por sexo**.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

idade_plot = df.loc[df['age'] < 200, 'age'].dropna()
axes[0, 0].hist(idade_plot, bins=40, color=CORES['primaria'], edgecolor='white', alpha=0.85, density=True)
sns.kdeplot(idade_plot, ax=axes[0, 0], color=CORES['secundaria'], linewidth=2)
axes[0, 0].set_title('Idade (anos; excl. código 300)')
axes[0, 0].set_xlabel('Idade')

w = df['weight'].dropna()
axes[0, 1].hist(w, bins=40, color=CORES['info'], edgecolor='white', alpha=0.85, density=True)
sns.kdeplot(w, ax=axes[0, 1], color=CORES['secundaria'], linewidth=2)
axes[0, 1].set_title('Peso (kg)')

h = df['height'].dropna()
axes[1, 0].hist(h, bins=35, color=CORES['ok'], edgecolor='white', alpha=0.85, density=True)
sns.kdeplot(h, ax=axes[1, 0], color=CORES['secundaria'], linewidth=2)
axes[1, 0].set_title('Altura (cm)')

df.boxplot(column='age', by='sex', ax=axes[1, 1])
axes[1, 1].set_title('Idade por sexo (0=M, 1=F)')
axes[1, 1].set_xlabel('Sexo')
plt.suptitle('')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig01_histogramas_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado:', FIG_DIR / 'fig01_histogramas_boxplot.png')


---
## 5. Testes de normalidade

**Shapiro–Wilk:** adequado a $n \lesssim 5000$; para idade usamos subamostra aleatória fixa (`seed=42`).  
**D'Agostino–Pearson** (`scipy.stats.normaltest`): teste omnibus sobre assimetria e curtose — útil com $n$ grande.  
**Kolmogorov–Smirnov** *versus* $\mathcal{N}(\hat\mu, \hat\sigma)$ **com parâmetros estimados nos mesmos dados**: o *p*-value não é nominal; reportamos como **exploratório** e priorizamos Q–Q e Shapiro/D'Agostino.

Variável-alvo principal: **idade** (sem código 300).


In [ ]:
idade = df.loc[df['age'] < 200, 'age'].dropna().astype(float)

# Shapiro (subamostra)
n_sh = min(5000, len(idade))
amostra = pd.Series(idade).sample(n=n_sh, random_state=42).values
stat_sw, p_sw = shapiro(amostra)
print(f'Shapiro–Wilk (idade, n={n_sh}): W={stat_sw:.5f}, p={p_sw:.2e}')

# D'Agostino
stat_k2, p_k2 = normaltest(idade)
print(f"D'Agostino–Pearson (idade, n={len(idade)}): K2={stat_k2:.3f}, p={p_k2:.2e}")

# KS exploratório (parâmetros ajustados)
mu, sig = idade.mean(), idade.std(ddof=1)
stat_ks, p_ks = kstest(idade, 'norm', args=(mu, sig))
print(f'KS vs N(mu_hat, sigma_hat) (exploratório): D={stat_ks:.5f}, p={p_ks:.2e}')

# Q-Q
fig, ax = plt.subplots(figsize=(6, 5))
stats.probplot(idade, dist='norm', plot=ax)
ax.set_title('Q–Q plot — idade vs. normal')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig02_qq_idade.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Homogeneidade de variâncias (Levene e Bartlett)

Comparamos **idade** e **peso** entre **dois grupos de sexo** (codificação PTB-XL: 0 e 1).  
**Levene** é robusto a desvios da normalidade; **Bartlett** assume normalidade aproximada nos grupos — reportamos ambos e interpretamos com cautela se as caudas forem pesadas.


In [ ]:
g0 = df.loc[df['sex'] == 0, 'age']
g1 = df.loc[df['sex'] == 1, 'age']
# Remover código 300 para estes testes
g0 = g0[g0 < 200].dropna()
g1 = g1[g1 < 200].dropna()

stat_lev, p_lev = levene(g0, g1, center='median')
print(f"Levene (idade: M vs F, center='median'): W={stat_lev:.4f}, p={p_lev:.4e}")

stat_bar, p_bar = bartlett(g0, g1)
print(f'Bartlett (idade: M vs F): K2={stat_bar:.4f}, p={p_bar:.4e}')

# Peso (homogeneidade de variâncias entre sexos)
w0 = df.loc[df['sex'] == 0, 'weight'].dropna()
w1 = df.loc[df['sex'] == 1, 'weight'].dropna()
stat_lev_w, p_lev_w = levene(w0, w1, center='median')
stat_bar_w, p_bar_w = bartlett(w0, w1)
print(f'Levene (peso: M vs F): W={stat_lev_w:.4f}, p={p_lev_w:.4e}')
print(f'Bartlett (peso: M vs F): K2={stat_bar_w:.4f}, p={p_bar_w:.4e}')


---
## 7. Dispositivos de aquisição (`device`)

O **Entregável 1** assinalou a heterogeneidade de modelos **Schiller** no PTB-XL. No E3, tratamos `device` como **fator de estratificação estatística**: descrevemos contagem por modelo, medianas de idade e peso, e testamos se as **distribuições de idade** diferem entre dispositivos (**Kruskal–Wallis**, $k$ amostras, sem assumir normalidade).

Também aplicamos **Levene** com `center='median'` à idade entre **todos** os grupos `device` (variâncias iguais?). **Caveat:** com muitos grupos e $n$ desiguais, um *p* baixo é esperado; o objetivo é **documentar** o efeito do equipamento, não concluir causalidade (o ano de registo confunde com o modelo).

Se existir o CSV do **E2**, acrescentamos **SQI mediano por registo** por `device` e um *boxplot* comparativo.


In [ ]:
vc_dev = df['device'].value_counts()
print('Modelos (device) e nº de registos:')
print(vc_dev.to_string())

ddev = df.reset_index()
ddev['age_clean'] = ddev['age'].where(ddev['age'] < 200)

agg_dev = ddev.groupby('device', observed=True).agg(
    n_registos=('ecg_id', 'count'),
    idade_mediana=('age_clean', 'median'),
    peso_mediana=('weight', 'median'),
)

if SQI_PATH.exists():
    sqi_long = pd.read_csv(SQI_PATH)
    sqi_rec = sqi_long.groupby('ecg_id', as_index=False)['sqi'].median()
    sqi_rec.columns = ['ecg_id', 'sqi_med']
    ddev = ddev.merge(sqi_rec, on='ecg_id', how='left')
    sq_m = ddev.groupby('device', observed=True)['sqi_med'].median()
    agg_dev = agg_dev.join(sq_m.rename('sqi_mediana'), how='left')

agg_dev = agg_dev.sort_values('n_registos', ascending=False)
print('\nTabela 2 — Estatísticas por dispositivo')
print(agg_dev.round(3).to_string())
agg_dev.to_csv(OUT_DIR / 'entregavel3_por_device.csv')

# Kruskal–Wallis: idade entre dispositivos
grupos_idade = [
    ddev.loc[ddev['device'] == dev, 'age_clean'].dropna()
    for dev in vc_dev.index
]
grupos_idade = [g for g in grupos_idade if len(g) >= 5]
if len(grupos_idade) >= 2:
    h_kw, p_kw = kruskal(*grupos_idade)
    print(f'\nKruskal–Wallis (idade entre {len(grupos_idade)} dispositivos com n≥5): H={h_kw:.3f}, p={p_kw:.2e}')

amostras_lev = [
    ddev.loc[ddev['device'] == dev, 'age_clean'].dropna().values
    for dev in vc_dev.index
]
amostras_lev = [a for a in amostras_lev if len(a) >= 2]
stat_lev_d, p_lev_d = levene(*amostras_lev, center='median')
print(f"Levene (idade, {len(vc_dev)} grupos device, center='median'): W={stat_lev_d:.4f}, p={p_lev_d:.2e}")

# Figura: idade e (se houver) SQI por device
ordem = vc_dev.index.tolist()
if SQI_PATH.exists():
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 6))
else:
    fig, ax0 = plt.subplots(figsize=(8, 6))
sns.boxplot(data=ddev, x='age_clean', y='device', order=ordem, ax=ax0, palette='Blues')
ax0.set_title('Idade por dispositivo (excl. código 300)')
ax0.set_xlabel('Idade (anos)')
if SQI_PATH.exists():
    sns.boxplot(data=ddev.dropna(subset=['sqi_med']), x='sqi_med', y='device', order=ordem, ax=ax1, palette='Oranges')
    ax1.set_title('SQI mediano por dispositivo (E2)')
    ax1.set_xlabel('SQI mediano (12 derivações)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig04_device_idade_sqi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado:', FIG_DIR / 'fig04_device_idade_sqi.png')


---
## 8. Correlações (Pearson e Spearman) e *heatmap*

Construímos uma matriz com variáveis numéricas **completas ou parcialmente observadas**: `age` (sem 300), `height`, `weight`, `sex`.  
Se o ficheiro do **Entregável 2** existir, acrescentamos **SQI mediano** e **SNR mediano** por `ecg_id`. A variável **`device`** é categórica; não entra na matriz Pearson/Spearman aqui — a comparação por equipamento está na **Secção 7**.

*Pearson* mede associação **linear**; *Spearman* mede relação **monotónica** (robusta a outliers).


In [ ]:
base = df[['age', 'height', 'weight', 'sex']].copy()
base.loc[base['age'] >= 200, 'age'] = np.nan

if SQI_PATH.exists():
    sqi_df = pd.read_csv(SQI_PATH)
    agg = sqi_df.groupby('ecg_id').agg(
        sqi_med=('sqi', 'median'),
        snr_med=('snr_db', 'median'),
    )
    base = base.join(agg, how='left')
    extra = ['sqi_med', 'snr_med']
else:
    extra = []

cols_corr = ['age', 'height', 'weight', 'sex'] + extra
print('Observações não nulas por variável:')
print(base[cols_corr].notna().sum())

R_pearson = base[cols_corr].corr(method='pearson', min_periods=30)
R_spearman = base[cols_corr].corr(method='spearman', min_periods=30)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(R_pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Pearson')
sns.heatmap(R_spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Spearman')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig03_heatmap_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

R_pearson.round(3).to_csv(OUT_DIR / 'entregavel3_matriz_pearson.csv')
R_spearman.round(3).to_csv(OUT_DIR / 'entregavel3_matriz_spearman.csv')
print('Matrizes guardadas em outputs/')


---
## 9. Síntese interpretativa

**Normalidade:** Para a idade, esperamos **rejeitar** $H_0$ de normalidade em amostras grandes (caudas, discretização, população hospitalar). Isso **não impede** o uso de métodos robustos ou de transformações no E4; informa a escolha de testes não paramétricos quando necessário.

**Homogeneidade:** Diferenças de variância da idade entre sexos, se significativas, sugerem estratificação ou modelos que permitam heterocedasticidade em análises futuras.

**Dispositivos:** Diferenças de idade e de SQI entre modelos `device` lembram que o dataset **não é i.i.d.** com respeito ao hardware; no **E4** e seguintes convém decidir se se **controla** o dispositivo (estratificação, covariável) ou se se assume como variabilidade realística.

**Correlações:** Associações fracas ou moderadas entre demografia e SQI são plausíveis — o SQI resume morfologia do traçado, não apenas idade.

**Ligação ao E4:** Resultados aqui justificam **limpeza e normalização** conscientes (e.g. *z-score* por estrato) antes de inferência pesada nos entregáveis seguintes.


---
## 10. Sumário do Entregável 3


In [ ]:
print('=' * 70)
print('ENTREGÁVEL 3 — ANÁLISE ESTATÍSTICA INICIAL')
print('=' * 70)
print('Ficheiros gerados:')
for p in sorted(FIG_DIR.glob('fig*.png')):
    print(' ', p.name)
for p in sorted(OUT_DIR.glob('entregavel3*.csv')):
    print(' ', p.name)
print('Próximo: E4 — limpeza; convém considerar `device` e listas de rejeição do E2 ao filtrar registos.')
